In [1]:
import pandas as pd

# Raw file stays unchanged; cleaning works on a loaded copy
df = pd.read_csv("../data/raw/netflix_titles.csv")

print(df.shape)
df.info()
print(df.isna().sum())

(8807, 12)
<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      6173 non-null   str  
 4   cast          7982 non-null   str  
 5   country       7976 non-null   str  
 6   date_added    8797 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8803 non-null   str  
 9   duration      8804 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes: int64(1), str(11)
memory usage: 3.9 MB
show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64


## Step 2: Cleaning rules

**Test used:** could this field have a value for this title? If yes, a blank means *unknown*. If no, it means *not applicable*. Missing values are kept as true missing values (not the text "NA") and are never filled with invented data.

| Column | Blanks | Meaning | Decision | Reason |
|---|---|---|---|---|
| director | 2,634 | Unknown | Keep as missing | Blank for 2,446 TV shows and 188 movies. Both can have directors, so the blanks are missing source data. Filling would invent names. |
| cast | 825 | Unknown | Keep as missing | Every title can have a cast. Filling would invent names. |
| country | 831 | Unknown | Keep as missing; excluded from country analysis | Every title can have a country. Filling would invent a location. |
| date_added | 10 | Unknown | Keep row; date stays missing | Only 10 of 8,807 rows. Other fields in these rows are still useful. |
| rating | 7 (was 4) | Unknown | Keep as missing | Three of these are the rows where the runtime was moved out. The true rating isn't in the data, so filling would invent one. |
| duration | 0 (was 3) | Unknown | Fixed | Three titles (Louis C.K. specials) had their runtime in the rating column. Moved to duration. |


In [2]:
# Remove stray spaces, then convert the text to a date
df["date_added"] = pd.to_datetime(df["date_added"].str.strip())

# Pull out the year as its own column for trend analysis
df["year_added"] = df["date_added"].dt.year

In [3]:
print(df["date_added"].dtype)
print(df["date_added"].isna().sum())
print(df[["date_added", "year_added"]].head())

datetime64[us]
10
  date_added  year_added
0 2021-09-25      2021.0
1 2021-09-24      2021.0
2 2021-09-24      2021.0
3 2021-09-24      2021.0
4 2021-09-24      2021.0


In [4]:
# Pull the number in front of "min" into a movie-only column
df["duration_mins"] = pd.to_numeric(df["duration"].str.extract(r"(\d+)\s*min")[0])

# Pull the number in front of "Season" into a show-only column
df["duration_seasons"] = pd.to_numeric(df["duration"].str.extract(r"(\d+)\s*Season")[0])

In [5]:
# Count non-blank values in each new column, split by type
df.groupby("type")[["duration_mins", "duration_seasons"]].count()

,duration_mins,duration_seasons
type,,
Movie,6128,0
TV Show,0,2676


In [6]:
# Show the rows where duration is blank, with the fields that might hold the value by mistake
df[df["duration"].isna()][["show_id", "type", "title", "rating", "duration"]]

,show_id,type,title,rating,duration
5541,s5542,Movie,Louis C.K. 2017,74 min,NaN
5794,s5795,Movie,Louis C.K.: Hilarious,84 min,NaN
5813,s5814,Movie,Louis C.K.: Live at the Comedy Store,66 min,NaN


In [7]:
# Rows where rating holds a runtime such as "74 min" (the duration was entered in the wrong column)
misplaced = df["rating"].str.contains(r"\d+\s*min", na=False)

# Move the runtime into duration, then clear rating because its real value isn't in the data
df.loc[misplaced, "duration"] = df.loc[misplaced, "rating"]
df.loc[misplaced, "rating"] = pd.NA

In [8]:
# Confirm the three rows now have a duration and a blank rating
print(df.loc[misplaced, ["title", "rating", "duration"]])
print(df["rating"].isna().sum())

                                     title rating duration
5541                       Louis C.K. 2017    NaN   74 min
5794                 Louis C.K.: Hilarious    NaN   84 min
5813  Louis C.K.: Live at the Comedy Store    NaN   66 min
7


In [9]:
# Check the three moved titles by name, which doesn't depend on the rating column
df[df["title"].str.contains("Louis C.K.", na=False)][["title", "rating", "duration"]]

,title,rating,duration
5541,Louis C.K. 2017,NaN,74 min
5794,Louis C.K.: Hilarious,NaN,84 min
5813,Louis C.K.: Live at the Comedy Store,NaN,66 min


In [10]:
# Rerun the split so the three moved movies are included
df["duration_mins"] = pd.to_numeric(df["duration"].str.extract(r"(\d+)\s*min")[0])
df["duration_seasons"] = pd.to_numeric(df["duration"].str.extract(r"(\d+)\s*Season")[0])

# Duration should now have no blanks
print(df["duration"].isna().sum())

0


In [11]:
# Remove leading and trailing spaces from every text column
text_cols = df.select_dtypes(include="object").columns
df[text_cols] = df[text_cols].apply(lambda col: col.str.strip())

C:\Users\Adedayo ONILEDE\AppData\Local\Temp\ipykernel_18648\316032642.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df.select_dtypes(include="object").columns


In [12]:
print(df.isna().sum())

show_id                0
type                   0
title                  0
director            2634
cast                 825
country              831
date_added            10
release_year           0
rating                 7
duration               0
listed_in              0
description            0
year_added            10
duration_mins       2676
duration_seasons    6131
dtype: int64


In [13]:
# Check for repeated show IDs, which should never happen
print(df["show_id"].duplicated().sum())

# Check for repeated title and type pairs
print(df.duplicated(subset=["title", "type"]).sum())

0
1


In [14]:
# Show every row that shares a title and type with another row
df[df.duplicated(subset=["title", "type"], keep=False)][
    ["show_id", "title", "type", "release_year", "country", "duration"]
]

,show_id,title,type,release_year,country,duration
3371,s3372,Consequences,Movie,2014,Turkey,106 min
6529,s6530,Consequences,Movie,2014,Turkey,106 min


In [15]:
# Compare the remaining text fields for the two matching rows
df[df["title"] == "Consequences"][["show_id", "director", "cast", "listed_in"]]

,show_id,director,cast,listed_in
3371,s3372,Ozan Açıktan,"Nehir Erdoğan, Tardu Flordun, İlker Kaleli, Se...","Dramas, International Movies, Thrillers"
6529,s6530,Ozan Açıktan,"Nehir Erdoğan, Tardu Flordun, İlker Kaleli, Se...","Dramas, International Movies, Thrillers"


In [16]:
rows_before = len(df)

# Keep the first copy of each title that matches on all the fields we compared
df = df.drop_duplicates(
    subset=["title", "type", "release_year", "country", "duration"],
    keep="first"
).reset_index(drop=True)

print(f"Removed {rows_before - len(df)} duplicate row(s). Rows now: {len(df)}")

Removed 1 duplicate row(s). Rows now: 8806


In [17]:
# Confirm the two Consequences rows match on the remaining fields before we drop one
print(df[df["title"] == "Consequences"][["show_id", "director", "cast", "listed_in"]])

     show_id      director                                               cast  \
3371   s3372  Ozan Açıktan  Nehir Erdoğan, Tardu Flordun, İlker Kaleli, Se...   

                                    listed_in  
3371  Dramas, International Movies, Thrillers  


In [18]:
# Read the untouched raw file and compare the two copies side by side
raw = pd.read_csv("../data/raw/netflix_titles.csv")
raw[raw["show_id"].isin(["s3372", "s6530"])][
    ["show_id", "title", "director", "cast", "country", "date_added", "release_year", "listed_in", "description"]
]

,show_id,title,director,cast,country,date_added,release_year,listed_in,description
3371,s3372,Consequences,Ozan Açıktan,"Nehir Erdoğan, Tardu Flordun, İlker Kaleli, Se...",Turkey,"October 25, 2019",2014,"Dramas, International Movies, Thrillers",Secrets bubble to the surface after a sensual ...
6529,s6530,Consequences,Ozan Açıktan,"Nehir Erdoğan, Tardu Flordun, İlker Kaleli, Se...",Turkey,"October 25, 2019",2014,"Dramas, International Movies, Thrillers",Secrets bubble to the surface after a sensual ...


In [19]:
# Show full text, not truncated, then compare the two descriptions exactly
pd.set_option("display.max_colwidth", None)

raw = pd.read_csv("../data/raw/netflix_titles.csv")
pair = raw[raw["show_id"].isin(["s3372", "s6530"])]

print(pair[["show_id", "description"]].to_string())
print("Descriptions identical:", pair["description"].iloc[0] == pair["description"].iloc[1])

     show_id                                                                                                                            description
3371   s3372  Secrets bubble to the surface after a sensual encounter and an unforeseen crime entangle two friends and a woman caught between them.
6529   s6530  Secrets bubble to the surface after a sensual encounter and an unforeseen crime entangle two friends and a woman caught between them.
Descriptions identical: True


**Duplicates:** "Consequences" (2014, Turkey, Ozan Açıktan) appears twice with identical values in every field, including the description. One copy (s6530) was removed.

In [20]:
# Save the cleaned table to the processed folder, leaving the raw file untouched
df.to_csv("../data/processed/netflix_clean.csv", index=False)

print(df.shape)

(8806, 15)
